Notebook 08 — Tuning and Significance

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import glob
import json
import time
import itertools

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier

from scipy import stats

import warnings
warnings.filterwarnings("ignore")


Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation"
FINAL_PROJECT_PATH = PROJECT_PATH + "/Final_Project"

SPLITS_PATH = FINAL_PROJECT_PATH + "/02_Data/Splits"
MODELS_PATH = FINAL_PROJECT_PATH + "/03_Models"
RESULTS_PATH = FINAL_PROJECT_PATH + "/04_Results"
PREDICTIONS_PATH = RESULTS_PATH + "/Predictions"
FINAL_TABLES_PATH = RESULTS_PATH + "/Final_Tables"

os.makedirs(os.path.join(MODELS_PATH, "RandomForest_tuned"), exist_ok=True)
os.makedirs(os.path.join(MODELS_PATH, "XGBoost_tuned"), exist_ok=True)
os.makedirs(FINAL_TABLES_PATH, exist_ok=True)

LABELS = ["Buy", "Hold", "Sell"]
RANDOM_STATE = 42
VAL_FRACTION = 0.20
# last 20% of TRAIN becomes validation -
RUN_SEARCH = True

N_SEARCH_CONFIGS = 6     # random search budget per model family
SEARCH_ROW_CAP = 12000   # rows used per stock during search only.
                         # The winning config is retrained on the FULL training set after.

# Two stocks from two different sectors - the search subset
SEARCH_STOCKS = ["Reliance", "HDFCB"]

train_files = sorted(glob.glob(os.path.join(SPLITS_PATH, "*_train.parquet")))
stock_names = [os.path.basename(f).replace("_train.parquet", "") for f in train_files]

_sample = pd.read_parquet(train_files[0])
EXCLUDE = {"datetime", "open", "high", "low", "close", "volume", "tick_count",
           "future_return", "target_threshold", "target_signal"}
FEATURE_COLUMNS = [c for c in _sample.columns if c not in EXCLUDE]

print(f"{len(stock_names)} stocks, {len(FEATURE_COLUMNS)} features")
print(f"Search subset: {SEARCH_STOCKS}")


10 stocks, 22 features
Search subset: ['Reliance', 'HDFCB']


---
# PART A — Hyperparameter search
---

In [ ]:
def load_train_val(stock, cap=None):
    df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_train.parquet"))
    cut = int(len(df) * (1 - VAL_FRACTION))
    tr, va = df.iloc[:cut], df.iloc[cut:]

    # Take the MOST RECENT "cap" training rows rather than a random sample -
    # keeps the block contiguous in time and closest to the validation period.
    if cap is not None and len(tr) > cap:
        tr = tr.iloc[-cap:]

    return (tr[FEATURE_COLUMNS], tr["target_signal"],
            va[FEATURE_COLUMNS], va["target_signal"])


# Random search spaces. Sampled rather than exhaustively gridded - random search
# covers a high-dimensional space more efficiently than grid search for the same budget.
rng = np.random.default_rng(RANDOM_STATE)

# NOTE: max_depth=None with small min_samples_leaf builds enormous trees on 46k rows
# (15+ minutes for a single fit). Bounded here so the search completes in reasonable time.
# Depth 16 with 400 trees is already far more capacity than this data supports.
# max_features=0.5 evaluates 11 features per split vs sqrt's ~5, roughly doubling
# fit time for no reliable gain on this data. Dropped, along with the deepest/largest
# combinations, to keep the search to a few minutes.
RF_SPACE = {
    "n_estimators": [100, 150, 200],
    "max_depth": [6, 8, 12],
    "min_samples_leaf": [50, 100, 200],
    "max_features": ["sqrt", "log2"],
}

XGB_SPACE = {
    "n_estimators": [200, 300],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 5, 20],
}


def sample_configs(space, n):
    keys = list(space.keys())
    seen, configs = set(), []
    attempts = 0
    while len(configs) < n and attempts < n * 50:
        attempts += 1
        cfg = {k: space[k][rng.integers(len(space[k]))] for k in keys}
        sig = tuple(sorted((k, str(v)) for k, v in cfg.items()))
        if sig not in seen:
            seen.add(sig)
            configs.append(cfg)
    return configs


rf_configs = sample_configs(RF_SPACE, N_SEARCH_CONFIGS)
xgb_configs = sample_configs(XGB_SPACE, N_SEARCH_CONFIGS)

print(f"Sampled {len(rf_configs)} RF configs and {len(xgb_configs)} XGBoost configs")
print(f"Total fits: {(len(rf_configs) + len(xgb_configs)) * len(SEARCH_STOCKS)}")


Sampled 6 RF configs and 6 XGBoost configs
Total fits: 24


In [ ]:
# Pre-load the search subset once, rather than re-reading parquet for every config
search_data = {s: load_train_val(s, cap=SEARCH_ROW_CAP) for s in SEARCH_STOCKS}
for s in SEARCH_STOCKS:
    Xtr, ytr, Xva, yva = search_data[s]
    print(f"{s:12s} train {Xtr.shape}  val {Xva.shape}")


Reliance     train (12000, 22)  val (11687, 22)
HDFCB        train (12000, 22)  val (11669, 22)


### Random Forest search

In [ ]:
rf_results = []
t_start = time.time()

if not RUN_SEARCH:
    print("RUN_SEARCH is False - skipping Random Forest search.")

for i, cfg in enumerate(rf_configs if RUN_SEARCH else [], 1):
    scores = []
    t_cfg = time.time()
    for s in SEARCH_STOCKS:
        Xtr, ytr, Xva, yva = search_data[s]
        model = RandomForestClassifier(
            **cfg, n_jobs=-1, random_state=RANDOM_STATE, class_weight="balanced_subsample"
        )
        model.fit(Xtr, ytr)
        scores.append(f1_score(yva, model.predict(Xva), average="macro", labels=LABELS, zero_division=0))

    rf_results.append({**cfg, "Val_F1_mean": np.mean(scores), "Val_F1_std": np.std(scores)})
    print(f"  [{i:2d}/{len(rf_configs)}] val F1 {np.mean(scores):.4f} +/- {np.std(scores):.4f}"
          f"  ({time.time()-t_cfg:.0f}s)  {cfg}")

if rf_results:
    rf_search = pd.DataFrame(rf_results).sort_values("Val_F1_mean", ascending=False).reset_index(drop=True)
    rf_search.to_csv(os.path.join(RESULTS_PATH, "HP_Search_RandomForest.csv"), index=False)
    print(f"\nRF search took {(time.time() - t_start)/60:.1f} min")
    print("\nTop configurations:")
    display(rf_search.head())
else:
    rf_search = None


  [ 1/6] val F1 0.3417 +/- 0.0324  (12s)  {'n_estimators': 100, 'max_depth': 12, 'min_samples_leaf': 100, 'max_features': 'sqrt'}
  [ 2/6] val F1 0.3380 +/- 0.0291  (17s)  {'n_estimators': 150, 'max_depth': 12, 'min_samples_leaf': 50, 'max_features': 'log2'}
  [ 3/6] val F1 0.3451 +/- 0.0279  (8s)  {'n_estimators': 100, 'max_depth': 6, 'min_samples_leaf': 100, 'max_features': 'log2'}
  [ 4/6] val F1 0.3451 +/- 0.0305  (14s)  {'n_estimators': 200, 'max_depth': 12, 'min_samples_leaf': 200, 'max_features': 'log2'}
  [ 5/6] val F1 0.3507 +/- 0.0297  (9s)  {'n_estimators': 150, 'max_depth': 6, 'min_samples_leaf': 200, 'max_features': 'sqrt'}
  [ 6/6] val F1 0.3401 +/- 0.0247  (13s)  {'n_estimators': 150, 'max_depth': 8, 'min_samples_leaf': 50, 'max_features': 'log2'}

RF search took 1.2 min

Top configurations:


,n_estimators,max_depth,min_samples_leaf,max_features,Val_F1_mean,Val_F1_std
0,150,6,200,sqrt,0.350726,0.029743
1,200,12,200,log2,0.345122,0.030506
2,100,6,100,log2,0.345071,0.027889
3,100,12,100,sqrt,0.341681,0.032401
4,150,8,50,log2,0.340113,0.024735


### XGBoost search

In [ ]:
LABEL_TO_INT = {l: i for i, l in enumerate(LABELS)}
xgb_results = []
t_start = time.time()

if not RUN_SEARCH:
    print("RUN_SEARCH is False - skipping XGBoost search.")

for i, cfg in enumerate(xgb_configs if RUN_SEARCH else [], 1):
    scores = []
    t_cfg = time.time()
    for s in SEARCH_STOCKS:
        Xtr, ytr, Xva, yva = search_data[s]
        model = XGBClassifier(
            **cfg, random_state=RANDOM_STATE, n_jobs=-1,
            tree_method="hist", eval_metric="mlogloss",
        )
        model.fit(Xtr, ytr.map(LABEL_TO_INT))
        pred = pd.Series(model.predict(Xva)).map({i: l for l, i in LABEL_TO_INT.items()}).values
        scores.append(f1_score(yva, pred, average="macro", labels=LABELS, zero_division=0))

    xgb_results.append({**cfg, "Val_F1_mean": np.mean(scores), "Val_F1_std": np.std(scores)})
    print(f"  [{i:2d}/{len(xgb_configs)}] val F1 {np.mean(scores):.4f} +/- {np.std(scores):.4f}"
          f"  ({time.time()-t_cfg:.0f}s)  {cfg}")

if xgb_results:
    xgb_search = pd.DataFrame(xgb_results).sort_values("Val_F1_mean", ascending=False).reset_index(drop=True)
    xgb_search.to_csv(os.path.join(RESULTS_PATH, "HP_Search_XGBoost.csv"), index=False)
    print(f"\nXGBoost search took {(time.time() - t_start)/60:.1f} min")
    print("\nTop configurations:")
    display(xgb_search.head())
else:
    xgb_search = None


  [ 1/6] val F1 0.3169 +/- 0.0177  (13s)  {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.03, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 5}
  [ 2/6] val F1 0.2866 +/- 0.0237  (3s)  {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 1}
  [ 3/6] val F1 0.3127 +/- 0.0205  (14s)  {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.6, 'min_child_weight': 20}
  [ 4/6] val F1 0.2928 +/- 0.0247  (5s)  {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.01, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 20}
  [ 5/6] val F1 0.3220 +/- 0.0170  (22s)  {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.6, 'colsample_bytree': 0.8, 'min_child_weight': 5}
  [ 6/6] val F1 0.3046 +/- 0.0234  (3s)  {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'subsample': 0.6, 'colsample_bytree': 1.0, 'min_child_

,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,Val_F1_mean,Val_F1_std
0,300,8,0.10,0.6,0.8,5,0.322005,0.016961
1,300,6,0.03,1.0,0.8,5,0.316941,0.017748
2,300,8,0.03,0.8,0.6,20,0.312663,0.020503
3,200,3,0.05,0.6,1.0,20,0.304619,0.023433
4,300,4,0.01,1.0,0.8,20,0.292779,0.024692


In [ ]:
def fix_types(cfg, int_keys):
    #CSV round-trip turns ints into floats and None into NaN - repairing before use
    out = {}
    for k, v in cfg.items():
        if k in int_keys:
            out[k] = None if (isinstance(v, float) and np.isnan(v)) else int(v)
        elif isinstance(v, str) or v is None:
            out[k] = v
        else:
            out[k] = float(v) if not float(v).is_integer() else int(v)
    return out


if rf_search is None or xgb_search is None:
    BEST_RF = BEST_XGB = None
    print("Search was skipped - no tuned configurations.")
    print("Part B (significance testing) still runs on the existing results below.")
else:
    BEST_RF = fix_types(
        {k: v for k, v in rf_search.iloc[0].items() if k not in ("Val_F1_mean", "Val_F1_std")},
        {"n_estimators", "max_depth", "min_samples_leaf"},
    )
    BEST_XGB = fix_types(
        {k: v for k, v in xgb_search.iloc[0].items() if k not in ("Val_F1_mean", "Val_F1_std")},
        {"n_estimators", "max_depth", "min_child_weight"},
    )

    print("Selected on validation:\n")
    print("Random Forest :", BEST_RF)
    print("XGBoost       :", BEST_XGB)

    print("\nPrevious defaults were:")
    print("Random Forest : {'n_estimators': 200, 'max_depth': 12, 'min_samples_leaf': 50, 'max_features': 'sqrt'}")
    print("XGBoost       : {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}")

    with open(os.path.join(RESULTS_PATH, "Best_Hyperparameters.json"), "w") as f:
        json.dump({"RandomForest": BEST_RF, "XGBoost": BEST_XGB}, f, indent=2, default=str)


Selected on validation:

Random Forest : {'n_estimators': 150, 'max_depth': 6, 'min_samples_leaf': 200, 'max_features': 'sqrt'}
XGBoost       : {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.1, 'subsample': 0.6, 'colsample_bytree': 0.8, 'min_child_weight': 5}

Previous defaults were:
Random Forest : {'n_estimators': 200, 'max_depth': 12, 'min_samples_leaf': 50, 'max_features': 'sqrt'}
XGBoost       : {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}


Retraining the winning configurations on all 10 stocks

In [ ]:
tuned_rows = []
t_start = time.time()

if BEST_RF is None:
    print("Skipping retraining - no tuned configuration available.")
else:
    for i, stock in enumerate(stock_names, 1):
        train_df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_train.parquet"))
        test_df = pd.read_parquet(os.path.join(SPLITS_PATH, f"{stock}_test.parquet"))

        Xtr, ytr = train_df[FEATURE_COLUMNS], train_df["target_signal"]
        Xte, yte = test_df[FEATURE_COLUMNS], test_df["target_signal"]

        out = test_df[["datetime", "close", "future_return", "target_signal"]].copy()

        rf = RandomForestClassifier(**BEST_RF, n_jobs=-1, random_state=RANDOM_STATE,
                                    class_weight="balanced_subsample")
        rf.fit(Xtr, ytr)
        rf_proba = rf.predict_proba(Xte)
        rf_pred = rf.predict(Xte)
        out["pred_RandomForest_tuned"] = rf_pred
        for k, lab in enumerate(rf.classes_):
            out[f"prob_RandomForest_tuned_{lab}"] = rf_proba[:, k]

        tuned_rows.append({
            "Stock": stock, "Model": "RandomForest_tuned",
            "Accuracy": round(accuracy_score(yte, rf_pred), 4),
            "F1_Macro": round(f1_score(yte, rf_pred, average="macro", labels=LABELS, zero_division=0), 4),
        })

        xgb = XGBClassifier(**BEST_XGB, random_state=RANDOM_STATE, n_jobs=-1,
                            tree_method="hist", eval_metric="mlogloss")
        xgb.fit(Xtr, ytr.map(LABEL_TO_INT))
        xgb_proba = xgb.predict_proba(Xte)
        xgb_pred = pd.Series(xgb_proba.argmax(axis=1)).map({v: k for k, v in LABEL_TO_INT.items()}).values
        out["pred_XGBoost_tuned"] = xgb_pred
        for k, lab in enumerate(LABELS):
            out[f"prob_XGBoost_tuned_{lab}"] = xgb_proba[:, k]

        tuned_rows.append({
            "Stock": stock, "Model": "XGBoost_tuned",
            "Accuracy": round(accuracy_score(yte, xgb_pred), 4),
            "F1_Macro": round(f1_score(yte, xgb_pred, average="macro", labels=LABELS, zero_division=0), 4),
        })

        out.to_parquet(os.path.join(PREDICTIONS_PATH, f"{stock}_tuned_predictions.parquet"), index=False)
        print(f"[{i:2d}/10] {stock:24s} RF {tuned_rows[-2]['F1_Macro']:.4f}   XGB {tuned_rows[-1]['F1_Macro']:.4f}")

    print(f"\nRetraining took {(time.time() - t_start)/60:.1f} min")

tuned_df = pd.DataFrame(tuned_rows)
if len(tuned_df):
    tuned_df.to_csv(os.path.join(RESULTS_PATH, "Tuned_Metrics_AllStocks.csv"), index=False)


[ 1/10] Eicher_Motors            RF 0.3592   XGB 0.3704
[ 2/10] HDFCB                    RF 0.3463   XGB 0.3601
[ 3/10] Hindustan_Unilever       RF 0.3713   XGB 0.3765
[ 4/10] ICICIBC                  RF 0.3614   XGB 0.3557
[ 5/10] INFOSYS                  RF 0.3740   XGB 0.3917
[ 6/10] ITC                      RF 0.3920   XGB 0.3649
[ 7/10] Mahindra_and_Mahindra    RF 0.3690   XGB 0.3566
[ 8/10] ONGC                     RF 0.3965   XGB 0.3582
[ 9/10] Reliance                 RF 0.3948   XGB 0.3634
[10/10] TCS                      RF 0.3837   XGB 0.3684

Retraining took 8.4 min


In [ ]:
# Did tuning actually change anything?
orig = pd.read_csv(os.path.join(RESULTS_PATH, "Model_Metrics_AllStocks.csv"))

cmp_rows = []
if len(tuned_df) == 0:
    print("No tuned results - skipping this comparison.")
for base, tuned in ([("RandomForest", "RandomForest_tuned"), ("XGBoost", "XGBoost_tuned")] if len(tuned_df) else []):
    o = orig[orig["Model"] == base]["F1_Macro"].mean()
    t = tuned_df[tuned_df["Model"] == tuned]["F1_Macro"].mean()
    cmp_rows.append({"Model": base, "Default_F1": round(o, 4), "Tuned_F1": round(t, 4),
                     "Change": round(t - o, 4)})

if cmp_rows:
    tuning_effect = pd.DataFrame(cmp_rows)
    tuning_effect.to_csv(os.path.join(FINAL_TABLES_PATH, "Table8_Tuning_Effect.csv"), index=False)
    print("Effect of hyperparameter tuning:\n")
    display(tuning_effect)
print("\nIf the change is small, that is itself worth reporting: it means the original")
print("defaults were near-optimal and the comparison was not unfairly disadvantaging the trees.")


Effect of hyperparameter tuning:



,Model,Default_F1,Tuned_F1,Change
0,RandomForest,0.3841,0.3748,-0.0093
1,XGBoost,0.3732,0.3666,-0.0066



If the change is small, that is itself worth reporting: it means the original
defaults were near-optimal and the comparison was not unfairly disadvantaging the trees.


---
# PART B — Statistical significance
---

### Assemble every model's per-stock F1 into one paired table

In [ ]:
frames = [orig[["Stock", "Model", "F1_Macro"]]]
if len(tuned_df):
    frames.append(tuned_df[["Stock", "Model", "F1_Macro"]])

for path in ["DL_Metrics_AllStocks.csv", "DL_Metrics_AllStocks_v2.csv"]:
    full = os.path.join(RESULTS_PATH, path)
    if os.path.exists(full):
        d = pd.read_csv(full)[["Stock", "Model", "F1_Macro"]].copy()
        if "v2" in path:
            d["Model"] = d["Model"] + "_v2"
        frames.append(d)
        print(f"included {path}")

allm = pd.concat(frames, ignore_index=True)
paired = allm.pivot_table(index="Stock", columns="Model", values="F1_Macro")
paired = paired.dropna(axis=1, how="any")     # only models scored on all 10 stocks

print(f"\nModels with complete results across all {len(paired)} stocks:")
print(list(paired.columns))
display(paired.round(4))


included DL_Metrics_AllStocks.csv
included DL_Metrics_AllStocks_v2.csv

Models with complete results across all 10 stocks:
['Baseline_Majority', 'Baseline_Momentum', 'Baseline_Random', 'LSTM', 'LSTM_v2', 'RandomForest', 'RandomForest_tuned', 'Transformer', 'Transformer_v2', 'XGBoost', 'XGBoost_tuned']


Model,Baseline_Majority,Baseline_Momentum,Baseline_Random,LSTM,LSTM_v2,RandomForest,RandomForest_tuned,Transformer,Transformer_v2,XGBoost,XGBoost_tuned
Stock,,,,,,,,,,,
Eicher_Motors,0.1757,0.2750,0.3357,0.2916,0.3022,0.3911,0.3592,0.3177,0.2672,0.3695,0.3704
HDFCB,0.1739,0.2794,0.3379,0.3193,0.3328,0.3650,0.3463,0.3498,0.3241,0.3620,0.3601
Hindustan_Unilever,0.1770,0.2802,0.3350,0.3876,0.3667,0.3739,0.3713,0.3654,0.3623,0.3798,0.3765
ICICIBC,0.1776,0.2847,0.3316,0.1929,0.2521,0.3683,0.3614,0.3573,0.2020,0.3613,0.3557
INFOSYS,0.1700,0.2755,0.3397,0.3765,0.3855,0.3849,0.3740,0.3690,0.3906,0.4034,0.3917
ITC,0.1812,0.2906,0.3387,0.3912,0.3844,0.3977,0.3920,0.3668,0.3823,0.3764,0.3649
Mahindra_and_Mahindra,0.1750,0.2801,0.3394,0.1616,0.1592,0.3740,0.3690,0.2746,0.1576,0.3723,0.3566
ONGC,0.1810,0.2861,0.3299,0.2651,0.2974,0.3913,0.3965,0.2452,0.2371,0.3707,0.3582
Reliance,0.1837,0.2862,0.3288,0.3849,0.3919,0.3998,0.3948,0.3622,0.3843,0.3611,0.3634


### The tests

For every pair of models: Wilcoxon signed-rank (primary), paired t-test (reference), and Cohen's d (effect size).

**Why paired tests.** Each model was measured on the same 10 stocks. Pairing removes between-stock variability — some stocks are simply harder than others — and tests only the model-to-model difference, which is what we care about.

In [ ]:
def cohens_d_paired(a, b):
    diff = np.asarray(a) - np.asarray(b)
    sd = diff.std(ddof=1)
    return diff.mean() / sd if sd > 0 else np.nan


models = list(paired.columns)
test_rows = []

for m1, m2 in itertools.combinations(models, 2):
    a, b = paired[m1].values, paired[m2].values
    diff = a - b

    if np.allclose(diff, 0):
        continue

    try:
        w_stat, w_p = stats.wilcoxon(a, b)
    except ValueError:
        w_stat, w_p = np.nan, np.nan

    t_stat, t_p = stats.ttest_rel(a, b)
    d = cohens_d_paired(a, b)

    test_rows.append({
        "Model_A": m1, "Model_B": m2,
        "Mean_A": round(a.mean(), 4), "Mean_B": round(b.mean(), 4),
        "Mean_Diff": round(diff.mean(), 4),
        "A_wins": int((diff > 0).sum()),
        "Wilcoxon_p": round(w_p, 5) if not np.isnan(w_p) else np.nan,
        "Ttest_p": round(t_p, 5),
        "Cohens_d": round(d, 3),
    })

tests = pd.DataFrame(test_rows)
print(f"{len(tests)} pairwise comparisons")


55 pairwise comparisons


In [ ]:
# Holm-Bonferroni correction. Testing many pairs inflates the chance that at least
# one comes out "significant" purely by luck; Holm controls that while being less
# conservative than plain Bonferroni.
def holm_bonferroni(pvals, alpha=0.05):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adjusted = np.empty(n)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (n - rank) * p[idx]
        running = max(running, val)          # enforce monotonicity
        adjusted[idx] = min(running, 1.0)
    return adjusted


tests["Wilcoxon_p_holm"] = holm_bonferroni(tests["Wilcoxon_p"].fillna(1.0)).round(5)
tests["Significant_5pct"] = tests["Wilcoxon_p_holm"] < 0.05

tests = tests.sort_values("Wilcoxon_p_holm").reset_index(drop=True)
tests.to_csv(os.path.join(FINAL_TABLES_PATH, "Table9_Significance_Tests.csv"), index=False)

print("Pairwise comparisons, sorted by corrected p-value.")
print("Cohen's d: 0.2 small, 0.5 medium, 0.8 large.\n")
display(tests)


Pairwise comparisons, sorted by corrected p-value.
Cohen's d: 0.2 small, 0.5 medium, 0.8 large.



,Model_A,Model_B,Mean_A,Mean_B,Mean_Diff,A_wins,Wilcoxon_p,Ttest_p,Cohens_d,Wilcoxon_p_holm,Significant_5pct
0,Baseline_Majority,Baseline_Momentum,0.1766,0.2815,-0.1049,0,0.00195,0.00000,-38.202,0.10725,False
1,Baseline_Majority,Baseline_Random,0.1766,0.3345,-0.1579,0,0.00195,0.00000,-21.550,0.10725,False
2,Baseline_Majority,RandomForest_tuned,0.1766,0.3748,-0.1982,0,0.00195,0.00000,-13.301,0.10725,False
3,Baseline_Majority,RandomForest,0.1766,0.3841,-0.2075,0,0.00195,0.00000,-17.301,0.10725,False
4,Baseline_Majority,Transformer,0.1766,0.3389,-0.1623,0,0.00195,0.00000,-3.482,0.10725,False
5,Baseline_Momentum,Baseline_Random,0.2815,0.3345,-0.0530,0,0.00195,0.00000,-7.014,0.10725,False
6,Baseline_Majority,XGBoost_tuned,0.1766,0.3666,-0.1900,0,0.00195,0.00000,-13.774,0.10725,False
7,Baseline_Majority,XGBoost,0.1766,0.3732,-0.1966,0,0.00195,0.00000,-12.796,0.10725,False
8,Baseline_Momentum,RandomForest_tuned,0.2815,0.3748,-0.0933,0,0.00195,0.00000,-6.461,0.10725,False
9,Baseline_Momentum,RandomForest,0.2815,0.3841,-0.1026,0,0.00195,0.00000,-8.182,0.10725,False


### The comparisons that matter most

In [ ]:
def show(m1, m2):
    row = tests[((tests["Model_A"] == m1) & (tests["Model_B"] == m2)) |
                ((tests["Model_A"] == m2) & (tests["Model_B"] == m1))]
    if len(row) == 0:
        print(f"  {m1} vs {m2}: not available")
        return
    r = row.iloc[0]
    verdict = "SIGNIFICANT" if r["Significant_5pct"] else "not significant"
    print(f"  {r['Model_A']} ({r['Mean_A']:.4f}) vs {r['Model_B']} ({r['Mean_B']:.4f})")
    print(f"     diff {r['Mean_Diff']:+.4f} | wins {r['A_wins']}/10 | "
          f"Wilcoxon p={r['Wilcoxon_p']} (Holm {r['Wilcoxon_p_holm']}) | d={r['Cohens_d']} -> {verdict}\n")

print("KEY COMPARISONS\n")
key_pairs = [
    ("RandomForest", "XGBoost"),
    ("RandomForest", "Baseline_Random"),
    ("RandomForest", "Baseline_Momentum"),
    ("RandomForest", "LSTM"),
    ("RandomForest", "Transformer"),
    ("RandomForest", "RandomForest_tuned"),
    ("LSTM", "Transformer"),
]
for m1, m2 in key_pairs:
    if m1 in models and m2 in models:
        show(m1, m2)


KEY COMPARISONS

  RandomForest (0.3841) vs XGBoost (0.3732)
     diff +0.0109 | wins 8/10 | Wilcoxon p=0.04883 (Holm 1.0) | d=0.657 -> not significant

  Baseline_Random (0.3345) vs RandomForest (0.3841)
     diff -0.0496 | wins 0/10 | Wilcoxon p=0.00195 (Holm 0.10725) | d=-3.296 -> not significant

  Baseline_Momentum (0.2815) vs RandomForest (0.3841)
     diff -0.1026 | wins 0/10 | Wilcoxon p=0.00195 (Holm 0.10725) | d=-8.182 -> not significant

  LSTM (0.3162) vs RandomForest (0.3841)
     diff -0.0679 | wins 1/10 | Wilcoxon p=0.01367 (Holm 0.42377) | d=-0.844 -> not significant

  RandomForest (0.3841) vs Transformer (0.3389)
     diff +0.0452 | wins 10/10 | Wilcoxon p=0.00195 (Holm 0.10725) | d=0.974 -> not significant

  RandomForest (0.3841) vs RandomForest_tuned (0.3748)
     diff +0.0093 | wins 9/10 | Wilcoxon p=0.01367 (Holm 0.42377) | d=0.919 -> not significant

  LSTM (0.3162) vs Transformer (0.3389)
     diff -0.0226 | wins 6/10 | Wilcoxon p=0.55664 (Holm 1.0) | d=-0.346 

In [ ]:
from math import sqrt

n = len(paired)
print(f"Sample size: n = {n} stocks\n")
print("Smallest effect detectable with 80% power at alpha=0.05 (paired t-test):")
for d in [0.5, 0.8, 1.0, 1.2, 1.5]:
    ncp = d * sqrt(n)
    crit = stats.t.ppf(0.975, n - 1)
    power = 1 - stats.nct.cdf(crit, n - 1, ncp) + stats.nct.cdf(-crit, n - 1, ncp)
    print(f"  Cohen's d = {d:.1f}  ->  power {power:.0%}")
print("\nInterpretation: with 10 stocks this study reliably detects only large effects.")
print("Small but real differences between models would likely go undetected.")


Sample size: n = 10 stocks

Smallest effect detectable with 80% power at alpha=0.05 (paired t-test):
  Cohen's d = 0.5  ->  power 29%
  Cohen's d = 0.8  ->  power 62%
  Cohen's d = 1.0  ->  power 80%
  Cohen's d = 1.2  ->  power 92%
  Cohen's d = 1.5  ->  power 99%

Interpretation: with 10 stocks this study reliably detects only large effects.
Small but real differences between models would likely go undetected.


In [ ]:
# PART C — Planned comparisons

import os, glob
import numpy as np, pandas as pd
from scipy import stats

RESULTS_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/04_Results"
FINAL_TABLES_PATH = RESULTS_PATH + "/Final_Tables"
os.makedirs(FINAL_TABLES_PATH, exist_ok=True)

# --- rebuild the paired per-stock F1 table from saved results ---
frames = [pd.read_csv(os.path.join(RESULTS_PATH, "Model_Metrics_AllStocks.csv"))[["Stock","Model","F1_Macro"]]]

for fname, suffix in [("Tuned_Metrics_AllStocks.csv", ""),
                      ("DL_Metrics_AllStocks.csv", ""),
                      ("DL_Metrics_AllStocks_v2.csv", "_v2")]:
    fp = os.path.join(RESULTS_PATH, fname)
    if os.path.exists(fp):
        d = pd.read_csv(fp)[["Stock","Model","F1_Macro"]].copy()
        if suffix:
            d["Model"] = d["Model"] + suffix
        frames.append(d)
        print("loaded", fname)

paired = pd.concat(frames, ignore_index=True) \
           .pivot_table(index="Stock", columns="Model", values="F1_Macro") \
           .dropna(axis=1, how="any")
print("\nmodels available:", list(paired.columns), "\n")

# --- helpers ---
def cohens_d_paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    sd = d.std(ddof=1)
    return d.mean() / sd if sd > 0 else np.nan

def holm_bonferroni(pvals):
    p = np.asarray(pvals, dtype=float); n = len(p)
    order = np.argsort(p); adj = np.empty(n); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (n - rank) * p[idx]); adj[idx] = min(run, 1.0)
    return adj

# --- planned comparisons: each answers a pre-stated research question ---
PLANNED = [
    ("RandomForest", "Baseline_Random",    "Q1  Does ML beat chance?"),
    ("RandomForest", "Baseline_Momentum",  "Q1  Does ML beat a trivial rule?"),
    ("RandomForest", "XGBoost",            "Q2  Which tree ensemble?"),
    ("RandomForest", "LSTM_v2",            "Q3  Trees vs recurrent"),
    ("RandomForest", "Transformer_v2",     "Q3  Trees vs attention"),
    ("LSTM_v2",      "Transformer_v2",     "Q4  Which DL architecture?"),
    ("RandomForest", "RandomForest_tuned", "Q5  Did tuning help?"),
]

rows = []
for m1, m2, q in PLANNED:
    if m1 not in paired.columns or m2 not in paired.columns:
        print(f"skipped (missing): {m1} vs {m2}")
        continue
    a, b = paired[m1].values, paired[m2].values
    diff = a - b
    _, wp = stats.wilcoxon(a, b)
    rows.append({"Question": q, "Model_A": m1, "Model_B": m2,
                 "Mean_A": round(a.mean(),4), "Mean_B": round(b.mean(),4),
                 "Mean_Diff": round(diff.mean(),4), "A_wins": int((diff>0).sum()),
                 "Wilcoxon_p": round(wp,5), "Cohens_d": round(cohens_d_paired(a,b),3)})

planned_df = pd.DataFrame(rows)
planned_df["p_holm"] = holm_bonferroni(planned_df["Wilcoxon_p"]).round(5)
planned_df["Significant_5pct"] = planned_df["p_holm"] < 0.05
planned_df.to_csv(os.path.join(FINAL_TABLES_PATH, "Table9b_Planned_Comparisons.csv"), index=False)

print(f"PLANNED COMPARISONS  ({len(planned_df)} tests, Holm-corrected within family)")
print("=" * 104)
for _, r in planned_df.iterrows():
    v = "SIGNIFICANT" if r["Significant_5pct"] else "not significant"
    print(f"{r['Question']:34s} {r['Model_A']:19s} {r['Mean_A']:.4f}  vs  {r['Model_B']:19s} {r['Mean_B']:.4f}")
    print(f"{'':34s}   diff {r['Mean_Diff']:+.4f} | wins {r['A_wins']}/10 | "
          f"p={r['Wilcoxon_p']:.5f} -> Holm {r['p_holm']:.4f} | d={r['Cohens_d']:+.2f}   {v}\n")

p_floor = 2 / 2 ** len(paired)
print(f"Significant: {int(planned_df['Significant_5pct'].sum())} of {len(planned_df)}")
print(f"\nWilcoxon floor at n={len(paired)}: p_min = {p_floor:.5f}")
print(f"Significance is only attainable with <= {int(0.05/p_floor)} tests in one family.")
print("The all-pairs table ran 55 - which is why nothing survived there.")

loaded Tuned_Metrics_AllStocks.csv
loaded DL_Metrics_AllStocks.csv
loaded DL_Metrics_AllStocks_v2.csv

models available: ['Baseline_Majority', 'Baseline_Momentum', 'Baseline_Random', 'LSTM', 'LSTM_v2', 'RandomForest', 'RandomForest_tuned', 'Transformer', 'Transformer_v2', 'XGBoost', 'XGBoost_tuned'] 

PLANNED COMPARISONS  (7 tests, Holm-corrected within family)
Q1  Does ML beat chance?           RandomForest        0.3841  vs  Baseline_Random     0.3345
                                     diff +0.0496 | wins 10/10 | p=0.00195 -> Holm 0.0137 | d=+3.30   SIGNIFICANT

Q1  Does ML beat a trivial rule?   RandomForest        0.3841  vs  Baseline_Momentum   0.2815
                                     diff +0.1026 | wins 10/10 | p=0.00195 -> Holm 0.0137 | d=+8.18   SIGNIFICANT

Q2  Which tree ensemble?           RandomForest        0.3841  vs  XGBoost             0.3732
                                     diff +0.0109 | wins 8/10 | p=0.04883 -> Holm 0.0977 | d=+0.66   not significant

Q3  Tr